# Baseline run (Base strategy)
**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order (click ▶ or press Shift+Enter).

In [1]:
# Check that a GPU is attached. You should see 'Tesla T4'.
!nvidia-smi

Sun Nov  8 14:00:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Settings
Use `mistral7b` while your Llama access is pending. Change to `llama8b` once Meta approves.

In [2]:
MODEL = "llama8b"   # options: mistral7b, llama8b

## 2. Get the code from your private GitHub repo
Paste a GitHub token when asked (github.com/settings/tokens → Generate new token (classic) → tick **repo** → Generate). It is hidden as you type and not saved in the notebook.

In [3]:
import getpass, os
gh_token = getpass.getpass("GitHub token: ")
if not os.path.exists("/content/project"):
    !git clone -q https://{gh_token}@github.com/faheemjabbar39-hub/llm-knowledge-integration.git /content/project
else:
    !git -C /content/project pull -q
del gh_token
%cd /content/project
!git log --oneline -1

GitHub token: ··········
/content/project
a62d199 (HEAD -> main, origin/main, origin/HEAD) Colab notebook: model setting, private-repo clone, resume notes


## 3. Install packages (~3 minutes)

In [4]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pytest -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kis (pyproject.toml) ... done
......                                                                   [100%]
6 passed in 1.86s


## 4. Log in to Hugging Face
Paste your Hugging Face **Read** token (starts with `hf_`).

In [5]:
from huggingface_hub import login
login()

## 5. Download the benchmarks and split them into train / test

In [6]:
!python scripts/01_prepare_benchmarks.py
!wc -l data/processed/*.jsonl

== hotpotqa
README.md: 100% 9.52k/9.52k [00:00<00:00, 5.01MB/s]

distractor/train-00000-of-00002.parquet: downloading bytes:   5% 7.67M/166M [00:01<00:23, 6.62MB/s]
distractor/train-00000-of-00002.parquet: downloading bytes:  19% 30.8M/166M [00:01<00:04, 30.7MB/s, 1.96MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes:  96% 160M/166M [00:02<00:00, 113MB/s, 13.4MB/s  ]
distractor/train-00000-of-00002.parquet: reconstructing file:  22% 36.5M/166M [00:02<00:07, 16.9MB/s,  703kB/s  ]
distractor/train-00000-of-00002.parquet: reconstructing file:  49% 80.7M/166M [00:02<00:01, 45.3MB/s, 3.35MB/s  ]
distractor/train-00000-of-00002.parquet: reconstructing file:  62% 103M/166M [00:02<00:01, 60.3MB/s, 7.44MB/s  ] 
distractor/train-00000-of-00002.parquet: reconstructing file:  84% 139M/166M [00:03<00:00, 83.3MB/s, 9.42MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes: 100% 166M/166M [00:03<00:00, 50.0MB/s, 14.8MB/s  ]
distractor/train-00000-of-00002.parquet: recons

## 6. Smoke test: 20 questions per dataset (~5 minutes incl. model download)

In [7]:
!python scripts/05_run_eval.py --model $MODEL --strategy base --limit 20
!python scripts/06_analyze.py

llama8b__base: 80 items, 0 already done, 80 to run
config.json: 100% 855/855 [00:00<00:00, 4.07MB/s]
tokenizer_config.json: 100% 55.4k/55.4k [00:00<00:00, 19.0MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 27.6MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.27MB/s]
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 74.4MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   4% 579M/16.1G [00:08<05:43, 45.1MB/s, 71.8MB/s  ]
Reconstructing (incomplete total...):   9% 1.41G/16.1G [00:18<02:22, 103MB/s,  109MB/s  ]
Reconstructing (incomplete total...):  10% 1.68G/16.1G [00:18<01:20, 178MB/s,  117MB/s  ]
Reconstructing (incomplete total

## 7. Full baseline (~1 hour on a T4)
If Colab disconnects, run cells 2-4 again and then this cell: it continues where it stopped.

In [8]:
!python scripts/05_run_eval.py --model $MODEL --strategy base

llama8b__base: 2000 items, 80 already done, 1920 to run
Loading weights: 100% 291/291 [01:04<00:00,  4.50it/s]
  0% 0/1920 [00:00<?, ?it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
100% 1920/1920 [43:12<00:00,  1.35s/it]
{
  "model": "llama8b",
  "hf_id": "meta-llama/Llama-3.1-8B-Instruct",
  "strategy": "base",
  "tag": "",
  "device": "cuda",
  "quant": "nf4",
  "n_items_this_run": 1920,
  "wall_clock_s": 2592.3,
  "peak_gpu_gb": 5.897
}


## 8. Results

In [9]:
!python scripts/06_analyze.py --gpu-usd-per-hour 0.35
import pandas as pd
pd.read_csv('results/summary.csv')

  model condition    dataset   n  accuracy  contains  hallucination_rate  latency_p50_ms  latency_p95_ms  latency_p99_ms  mean_latency_ms  peak_gpu_gb  cost_per_1k_usd
llama8b      base   hotpotqa 500     0.162     0.200                 NaN         1351.19       1718.4685       2535.7474        1376.3835        5.897           0.1338
llama8b      base   mmlu_fin 500     0.578     0.578                 NaN         1172.13       1570.4155       1911.6312        1284.7114        5.897           0.1249
llama8b      base         nq 500     0.196     0.272                 NaN         1330.83       2214.9615       4584.6816        1457.3151        5.897           0.1417
llama8b      base truthfulqa 500     0.540     0.540                0.46         1163.20       1555.8790       1935.3097        1259.1745        5.897           0.1224

Only one condition per model so far - paired tests run once a second strategy exists.


,model,condition,dataset,n,accuracy,contains,hallucination_rate,latency_p50_ms,latency_p95_ms,latency_p99_ms,mean_latency_ms,peak_gpu_gb,cost_per_1k_usd
0,llama8b,base,hotpotqa,500,0.162,0.200,NaN,1351.19,1718.4685,2535.7474,1376.3835,5.897,0.1338
1,llama8b,base,mmlu_fin,500,0.578,0.578,NaN,1172.13,1570.4155,1911.6312,1284.7114,5.897,0.1249
2,llama8b,base,nq,500,0.196,0.272,NaN,1330.83,2214.9615,4584.6816,1457.3151,5.897,0.1417
3,llama8b,base,truthfulqa,500,0.540,0.540,0.46,1163.20,1555.8790,1935.3097,1259.1745,5.897,0.1224


## 9. Save results to Google Drive (Colab deletes files when the session ends)

In [11]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kis_results
!cp -r results/* /content/drive/MyDrive/kis_results/
!ls /content/drive/MyDrive/kis_results/raw

Mounted at /content/drive
llama8b__base.csv  llama8b__base.meta.json
